# Chapter 4: Naïve Bayesian Classification

This chapter introduces the **Naïve Bayesian classifier**, a simple probabilistic model based on **Bayes' theorem** and the assumption of strong independence among features (class conditional independence). The goal is to classify an instance $X = (x_1, x_2, \dots, x_n)$ by calculating the posterior probability $P(c_j|X)$ for each class $c_j$ and assigning $X$ to the class that maximizes this probability.

$$P(c_j|X) \propto P(c_j) \times P(X|c_j)$$
Where the likelihood $P(X|c_j)$ is simplified under the Naïve assumption: 
$$P(X|c_j) = \prod_{i} P(x_i|c_j)$$


## 1. Data Setup and Loading

Chúng ta sẽ sử dụng tập dữ liệu mẫu được sử dụng trong ví dụ của Chương 4 để dự đoán liệu có chơi Tennis hay không, với 14 bản ghi và 4 thuộc tính đầu vào (Outlook, Temperature, Humidity, Windy) và một biến lớp (Play).

*(Lưu ý: Vì không có file `data.csv`, chúng ta tạo dữ liệu thủ công bằng thư viện `pandas` để mô phỏng việc tải dữ liệu.)*

In [3]:
import pandas as pd
import numpy as np
from collections import defaultdict

# Dữ liệu mẫu (14 bản ghi) theo Bảng 4.1
data = {
    'Outlook': ['sunny', 'sunny', 'overcast', 'rainy', 'rainy', 'rainy', 'overcast', 'sunny', 'sunny', 'rainy', 'sunny', 'overcast', 'overcast', 'rainy'],
    'Temperature': ['hot', 'hot', 'hot', 'mild', 'cool', 'cool', 'cool', 'mild', 'cool', 'mild', 'mild', 'mild', 'hot', 'mild'],
    'Humidity': ['high', 'high', 'high', 'high', 'normal', 'normal', 'normal', 'high', 'normal', 'normal', 'normal', 'high', 'normal', 'high'],
    'Windy': ['false', 'true', 'false', 'false', 'false', 'true', 'true', 'false', 'false', 'false', 'true', 'true', 'false', 'true'],
    'Play': ['no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'no']
}

df = pd.DataFrame(data)
tot_rec = len(df) # Tổng số bản ghi là 14

print(f"Total records: {tot_rec}")
print("Sample Data:")
print(df)

Total records: 14
Sample Data:
     Outlook Temperature Humidity  Windy Play
0      sunny         hot     high  false   no
1      sunny         hot     high   true   no
2   overcast         hot     high  false  yes
3      rainy        mild     high  false  yes
4      rainy        cool   normal  false  yes
5      rainy        cool   normal   true   no
6   overcast        cool   normal   true  yes
7      sunny        mild     high  false   no
8      sunny        cool   normal  false  yes
9      rainy        mild   normal  false  yes
10     sunny        mild   normal   true  yes
11  overcast        mild     high   true  yes
12  overcast         hot   normal  false  yes
13     rainy        mild     high   true   no


## 2. Prior Probability (Xác suất Tiên nghiệm)

Tính xác suất tiên nghiệm $P(c_j)$ cho mỗi lớp, dựa trên kinh nghiệm từ tập dữ liệu huấn luyện.

Trong ví dụ này, có 9 bản ghi 'Yes' và 5 bản ghi 'No'.
$$P(\text{Play=Yes}) = 9/14 \quad P(\text{Play=No}) = 5/14$$


In [4]:
class_counts = df['Play'].value_counts() # Đếm số lần xuất hiện của mỗi lớp
prior_probabilities = class_counts / tot_rec # Tính P(Class)

classes = list(class_counts.index) # Các lớp duy nhất: 'yes', 'no'
nc = len(classes) # Số lượng lớp (2)

print(f"Class Counts:\n{class_counts}")
print(f"Prior Probabilities (P(Play)):\n{prior_probabilities}")

Class Counts:
Play
yes    9
no     5
Name: count, dtype: int64
Prior Probabilities (P(Play)):
Play
yes    0.642857
no     0.357143
Name: count, dtype: float64


## 3. Likelihood Calculation (Tính toán Khả năng)

Tính các xác suất điều kiện (likelihood) $P(x_i|c_j)$ cho mỗi thuộc tính $x_i$ biết lớp $c_j$.

### Áp dụng Laplace Estimator (Ước lượng Laplace)

Nếu một sự kết hợp thuộc tính/lớp không bao giờ xuất hiện trong tập huấn luyện (ví dụ: Outlook=Overcast khi Play=No), xác suất điều kiện sẽ là 0, điều này làm cho toàn bộ xác suất hậu nghiệm bằng 0. Để xử lý vấn đề này, ta sử dụng **Ước lượng Laplace** (Laplace Estimator):
$$\text{Smoothed Probability} = \frac{\text{Count}(x_i, c_j) + 1}{\text{Count}(c_j) + |V_i|}$$
Trong đó $|V_i|$ là số lượng giá trị duy nhất của thuộc tính $i$ (đóng vai trò là hệ số làm mịn $m$).

*(Lưu ý: Đối với Prior Probability, công thức Laplace cũng được áp dụng theo nguồn: $(N_c + 1) / (N_{total} + 2)$.)*

In [5]:
# Tính toán Likelihood (Bảng xác suất điều kiện) có áp dụng Laplace Smoothing

likelihood_tables = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))

features = df.columns[:-1] # Lấy tên các thuộc tính (trừ cột 'Play')

for feature in features:
    unique_values_feature = df[feature].nunique() # Số lượng giá trị duy nhất của thuộc tính
    
    # Tính tần suất kết hợp giữa Feature và Class
    for class_name in classes:
        subset = df[df['Play'] == class_name]
        class_count = len(subset) # Count(c_j)
        
        # Tính xác suất đã được làm mịn (Smoothed Likelihood)
        for value in df[feature].unique():
            # Count(x_i, c_j) + 1
            count_xi_cj = len(subset[subset[feature] == value]) + 1 
            
            # Count(c_j) + |V_i|
            denominator = class_count + unique_values_feature 
            
            likelihood_tables[feature][value][class_name] = count_xi_cj / denominator
            
print("Smoothed Likelihood for Outlook:")
print(pd.DataFrame(likelihood_tables['Outlook']))

# Áp dụng Laplace cho Prior Probability (P(C))
smoothed_prior = (class_counts + 1) / (tot_rec + nc)
print(f"\nSmoothed Prior Probabilities:\n{smoothed_prior}")

Smoothed Likelihood for Outlook:
     sunny  overcast     rainy
yes   0.25  0.416667  0.333333
no    0.50  0.125000  0.375000

Smoothed Prior Probabilities:
Play
yes    0.625
no     0.375
Name: count, dtype: float64


## 4. Posterior Prediction (Dự đoán Hậu nghiệm)

Dự đoán lớp cho một bản ghi mới $X$. Trong ví dụ của Chương 4, bản ghi cần phân loại là:
$$X = (\text{Outlook} = \text{sunny}, \text{Temperature} = \text{hot}, \text{Humidity} = \text{high}, \text{Windy} = \text{false})$$

Ta so sánh tử số của Xác suất Hậu nghiệm, vốn tỉ lệ với Tích (Prior $\times$ Likelihood):
$$P(c_j|X) \propto P(c_j) \times \prod_{i} P(x_i|c_j)$$

In [7]:
A = {'Outlook': 'sunny', 'Temperature': 'hot', 'Humidity': 'high', 'Windy': 'false'} # Bản ghi X cần dự đoán
results = {}

for class_name in classes:
    # Khởi tạo tích likelihood bằng Prior Probability P(C)
    posterior_numerator = smoothed_prior[class_name]
    
    for feature, value in A.items():
        # Lấy xác suất điều kiện P(x_i | c_j) đã được làm mịn
        likelihood = likelihood_tables[feature][value][class_name]
        posterior_numerator *= likelihood
    
    results[class_name] = posterior_numerator

ProbN = results.get('no') # P(Play=No | X) proportional
ProbP = results.get('yes') # P(Play=Yes | X) proportional

print(f"P(Play=No | X) proportional: {ProbN}")
print(f"P(Play=Yes | X) proportional: {ProbP}")

# So sánh để đưa ra dự đoán
if ProbN > ProbP:
    prediction = 'N' # Dự đoán là No
else:
    prediction = 'P' # Dự đoán là Yes

print(f"Prediction: {prediction}")
print(f"Maximum posterior numerator corresponds to class: {max(results, key=results.get)}")

P(Play=No | X) proportional: 0.021524234693877552
P(Play=Yes | X) proportional: 0.009039256198347109
Prediction: N
Maximum posterior numerator corresponds to class: no
